# 05 · Final evaluation on Spider dev

**This is the only notebook that touches dev.** Run it once per model, after every decision has been made on val.

In [ ]:
# --- setup: run once per session -------------------------------------------
import os, subprocess, sys
SPIDER_DIR = "/kaggle/input/spider/spider"   # <- EDIT: the folder with train_spider.json and database/
RUNS = "/kaggle/working/runs"
os.environ["SPIDER_DIR"], os.environ["MIRAGE_RUNS"] = SPIDER_DIR, RUNS
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
os.chdir("/kaggle/working/mirage")
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# If you are continuing from an earlier notebook, copy its runs folder in:
# !cp -r /kaggle/input/<your-previous-notebook-output>/runs/* /kaggle/working/runs/ 2>/dev/null || true


In [ ]:
!mirage generate -c configs/grpo_base.yaml --split dev --label start --out $MIRAGE_RUNS/eval/start-dev
!mirage generate -c configs/grpo_base.yaml --split dev --adapter $MIRAGE_RUNS/arms/single/adapter --label grpo-single --out $MIRAGE_RUNS/eval/grpo-single-dev
!mirage generate -c configs/grpo_base.yaml --split dev --adapter $MIRAGE_RUNS/arms/suite/adapter --label grpo-suite --out $MIRAGE_RUNS/eval/grpo-suite-dev

In [ ]:
for name in ['start', 'grpo-single', 'grpo-suite']:
    !mirage score $MIRAGE_RUNS/eval/{name}-dev/predictions.jsonl --spider $SPIDER_DIR --suites $MIRAGE_RUNS/suites --label {name}

Optional: official test-suite accuracy. Clone `taoyds/test-suite-sql-eval`, download its test-suite databases, and pass `--official <repo> --official-db <databases>` to `mirage score`.

In [ ]:
!cp configs/study.example.yaml $MIRAGE_RUNS/../study.yaml && sed -i 's#\.\./runs#runs#g' $MIRAGE_RUNS/../study.yaml
!cd $MIRAGE_RUNS/.. && mirage report study.yaml